In [ ]:
import os
import pickle
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Batch
from torch_geometric.nn import GATConv, global_mean_pool
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import f1_score, roc_auc_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Configuration
DATA_DIR = "GothamDataset2025/processed_features/gotham_graphs"
CHECKPOINT_DIR = os.path.join(DATA_DIR, "ssl_checkpoints")
ENCODER_PATH = os.path.join(CHECKPOINT_DIR, "ssl_encoder_best.pth")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load graph data
def load_graphs(split):
    with open(os.path.join(DATA_DIR, f"{split}_graphs.pkl"), 'rb') as f:
        return pickle.load(f)

train_graphs = load_graphs('train')
print(f"Loaded {len(train_graphs)} training graphs.")

# Build label mapping from training set
def get_class_mapping(graphs):
    attack_names = set()
    for g in graphs:
        for name in g.edge_attack_names:
            if name != 'Benign':
                attack_names.add(name)
    attack_names = sorted(list(attack_names))
    class_to_idx = {name: i for i, name in enumerate(attack_names)}
    num_classes = len(attack_names)
    return attack_names, class_to_idx, num_classes

attack_names, class_to_idx, num_classes = get_class_mapping(train_graphs)
print(f"\nAttack classes: {attack_names}")
print(f"Number of classes (including benign): {num_classes + 1}")

def assign_labels(graphs, class_to_idx, num_classes):
    # We'll assign both binary and multi‑hot labels
    for g in graphs:
        if not hasattr(g, 'y_multihot'):
            # Multi‑hot: one per attack class
            labels = torch.zeros(num_classes, dtype=torch.float)
            for name in g.edge_attack_names:
                if name != 'Benign' and name in class_to_idx:
                    labels[class_to_idx[name]] = 1.0
            g.y_multihot = labels.unsqueeze(0)
        if not hasattr(g, 'y_binary'):
            g.y_binary = torch.tensor(
                1 if any(name != 'Benign' for name in g.edge_attack_names) else 0,
                dtype=torch.float
            ).unsqueeze(0)
        # Also create a single‑class label for multi‑class classification:
        # If a graph has multiple attacks, we'll treat it as "mixed" – but for simplicity,
        # we'll use the first attack class as the label (or we can treat each class separately).
        # For multi‑class separability, we'll create a label that is 0 for benign,
        # and for attacks we'll assign the index of the first attack (or use multi‑label).
    return graphs

assign_labels(train_graphs, class_to_idx, num_classes)

# Build a multi‑class label: 0 = benign, else first attack class index
multi_class_labels = []
for g in train_graphs:
    if g.y_binary.item() == 0:
        multi_class_labels.append(0)  # benign
    else:
        # Get indices of all attacks present
        attack_indices = torch.where(g.y_multihot.squeeze() == 1)[0]
        # Use the first attack class as label (or we could use the most frequent)
        # For better multi‑class, we could also create a "mixed" class, but we'll keep it simple.
        multi_class_labels.append(int(attack_indices[0]) + 1)  # +1 because 0 is benign
multi_class_labels = np.array(multi_class_labels)
num_multi_classes = num_classes + 1  # benign + attack classes

print(f"Multi‑class labels: {num_multi_classes} classes (0=benign, 1..{num_classes}=attack types)")

# Build GAT encoder (same architecture)
class GATEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels, num_heads, num_layers, edge_dim=0, dropout=0.0):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        self.dropout = dropout
        self.edge_dim = edge_dim
        self.convs.append(GATConv(in_channels, hidden_channels, heads=num_heads,
                                  edge_dim=edge_dim, concat=True, dropout=dropout))
        self.bns.append(nn.BatchNorm1d(hidden_channels * num_heads))
        for _ in range(num_layers - 2):
            self.convs.append(GATConv(hidden_channels * num_heads, hidden_channels, heads=num_heads,
                                      edge_dim=edge_dim, concat=True, dropout=dropout))
            self.bns.append(nn.BatchNorm1d(hidden_channels * num_heads))
        self.convs.append(GATConv(hidden_channels * num_heads, hidden_channels, heads=1,
                                  edge_dim=edge_dim, concat=False, dropout=dropout))
        self.bns.append(nn.BatchNorm1d(hidden_channels))

    def forward(self, x, edge_index, edge_attr=None, batch=None):
        x = torch.nan_to_num(x, nan=0.0, posinf=1e3, neginf=-1e3)
        if edge_attr is not None:
            edge_attr = torch.nan_to_num(edge_attr, nan=0.0, posinf=1e3, neginf=-1e3)
        for i, conv in enumerate(self.convs):
            if self.edge_dim > 0 and edge_attr is not None:
                x = conv(x, edge_index, edge_attr=edge_attr)
            else:
                x = conv(x, edge_index)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = global_mean_pool(x, batch)
        return torch.nan_to_num(x, nan=0.0, posinf=1e3, neginf=-1e3)

# Load the pretrained encoder
sample = train_graphs[0]
node_dim = sample.x.size(1)
edge_dim = sample.edge_attr.size(1) if sample.edge_attr is not None else 0

encoder = GATEncoder(node_dim, hidden_channels=128, num_heads=4, num_layers=3,
                     edge_dim=edge_dim, dropout=0.2).to(DEVICE)
encoder.load_state_dict(torch.load(ENCODER_PATH, map_location=DEVICE))
encoder.eval()
print(f"\nLoaded SSL encoder from {ENCODER_PATH}")

# Extract embeddings
embeddings = []
binary_labels = []
multi_class_labels_list = []
batch_size = 32

for i in tqdm(range(0, len(train_graphs), batch_size), desc="Extracting embeddings"):
    batch_graphs = train_graphs[i:i+batch_size]
    bg = Batch.from_data_list(batch_graphs).to(DEVICE)
    with torch.no_grad():
        emb = encoder(bg.x, bg.edge_index, bg.edge_attr, bg.batch)
    embeddings.append(emb.cpu().numpy())
    for g in batch_graphs:
        binary_labels.append(g.y_binary.item())
        # multi‑class label (first attack class, 0 for benign)
        if g.y_binary.item() == 0:
            multi_class_labels_list.append(0)
        else:
            attack_indices = torch.where(g.y_multihot.squeeze() == 1)[0]
            multi_class_labels_list.append(int(attack_indices[0]) + 1)

embeddings = np.vstack(embeddings)
binary_labels = np.array(binary_labels)
multi_class_labels = np.array(multi_class_labels_list)

print(f"\nEmbeddings shape: {embeddings.shape}")
print(f"Binary: {np.sum(binary_labels)} attacks out of {len(binary_labels)}")
print(f"Multi‑class distribution:")
for c in range(num_multi_classes):
    count = np.sum(multi_class_labels == c)
    name = "Benign" if c == 0 else attack_names[c-1]
    print(f"  {name}: {count} samples")

# Scale embeddings for linear models
scaler = StandardScaler()
embeddings_scaled = scaler.fit_transform(embeddings)

# 1. Binary separability (attack vs benign)
X_train, X_test, y_train, y_test = train_test_split(embeddings_scaled, binary_labels, test_size=0.2, random_state=42)
clf_bin = LogisticRegression(max_iter=1000, class_weight='balanced')
clf_bin.fit(X_train, y_train)
y_pred_bin = clf_bin.predict(X_test)
y_prob_bin = clf_bin.predict_proba(X_test)[:, 1]
bin_f1 = f1_score(y_test, y_pred_bin)
bin_auc = roc_auc_score(y_test, y_prob_bin)
print(f"\n=== Binary Classification (Benign vs Attack) ===")
print(f"F1: {bin_f1:.4f}")
print(f"AUC: {bin_auc:.4f}")

# 2. Multi‑class separability (Benign + each attack type)
# We'll use one‑vs‑rest for each class to get per‑class AUC and F1.
per_class_f1 = {}
per_class_auc = {}
per_class_recall = {}
per_class_precision = {}

print(f"\n=== Per‑Class One‑vs‑Rest Performance ===")
print(f"{'Class':20} {'F1':8} {'AUC':8} {'Recall':8} {'Precision':8}")
for class_idx in range(num_multi_classes):
    # Create binary labels for this class vs rest
    y_class = (multi_class_labels == class_idx).astype(int)
    # Skip if class has too few samples
    if np.sum(y_class) < 5:
        print(f"{'Benign' if class_idx==0 else attack_names[class_idx-1]:20} {'(too few samples)'}")
        continue
    X_tr, X_te, y_tr, y_te = train_test_split(embeddings_scaled, y_class, test_size=0.2, random_state=42)
    clf = LogisticRegression(max_iter=1000, class_weight='balanced')
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te)
    y_prob = clf.predict_proba(X_te)[:, 1]
    f1 = f1_score(y_te, y_pred)
    auc = roc_auc_score(y_te, y_prob)
    recall = recall_score(y_te, y_pred)
    precision = precision_score(y_te, y_pred)
    name = "Benign" if class_idx == 0 else attack_names[class_idx-1]
    per_class_f1[name] = f1
    per_class_auc[name] = auc
    per_class_recall[name] = recall
    per_class_precision[name] = precision
    print(f"{name:20} {f1:8.4f} {auc:8.4f} {recall:8.4f} {precision:8.4f}")

# 3. Multi‑class classification (directly) using softmax
# This gives a holistic view of how well the encoder separates all classes.
# We'll use a simple MLP or logistic regression with multi‑class
from sklearn.linear_model import LogisticRegression
clf_multi = LogisticRegression(max_iter=1000, multi_class='multinomial', solver='lbfgs', class_weight='balanced')
X_tr, X_te, y_tr, y_te = train_test_split(embeddings_scaled, multi_class_labels, test_size=0.2, random_state=42)
clf_multi.fit(X_tr, y_tr)
y_pred_multi = clf_multi.predict(X_te)
multi_f1_macro = f1_score(y_te, y_pred_multi, average='macro')
multi_f1_weighted = f1_score(y_te, y_pred_multi, average='weighted')
print(f"\n=== Multi‑Class Classification (Benign + Attack Types) ===")
print(f"Macro F1: {multi_f1_macro:.4f}")
print(f"Weighted F1: {multi_f1_weighted:.4f}")
# Classification report
print("\nClassification Report (per class):")
print(classification_report(y_te, y_pred_multi, target_names=['Benign']+attack_names, zero_division=0))

# 4. t‑SNE visualisation (multi‑class)
try:
    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    # Sample a subset for speed
    sample_size = min(500, len(embeddings))
    idx = np.random.choice(len(embeddings), sample_size, replace=False)
    emb_sample = embeddings[idx]
    labels_sample = multi_class_labels[idx]
    
    emb_2d = tsne.fit_transform(emb_sample)
    plt.figure(figsize=(10,8))
    # Colour mapping
    cmap = plt.cm.get_cmap('tab10', num_multi_classes)
    for c in range(num_multi_classes):
        mask = labels_sample == c
        name = "Benign" if c == 0 else attack_names[c-1]
        plt.scatter(emb_2d[mask,0], emb_2d[mask,1], c=[cmap(c)], label=name, alpha=0.6, s=10)
    plt.title("t-SNE of SSL Encoder Embeddings (Multi‑Class)")
    plt.legend(loc='best', fontsize=8)
    plt.savefig("ssl_encoder_tsne_multiclass.png", dpi=150)
    print("\nSaved multi‑class t-SNE plot to ssl_encoder_tsne_multiclass.png")
except Exception as e:
    print(f"t-SNE failed: {e}")

# 5. Centroids and intra‑class variance
print("\n=== Class Centroids and Intra‑class Variance ===")
# Compute centroids for each class
centroids = {}
variances = {}
for c in range(num_multi_classes):
    mask = multi_class_labels == c
    if np.sum(mask) == 0:
        continue
    class_emb = embeddings[mask]
    centroid = class_emb.mean(axis=0)
    var = np.mean([np.linalg.norm(x - centroid) for x in class_emb])
    centroids[c] = centroid
    variances[c] = var
    name = "Benign" if c == 0 else attack_names[c-1]
    print(f"{name:20} centroid norm: {np.linalg.norm(centroid):.4f}, intra‑var: {var:.4f}")

# Compute inter‑centroid distances
print("\nInter‑centroid distances (Euclidean):")
for i in range(min(5, num_multi_classes)):
    for j in range(i+1, min(5, num_multi_classes)):
        if i in centroids and j in centroids:
            dist = np.linalg.norm(centroids[i] - centroids[j])
            name_i = "Benign" if i == 0 else attack_names[i-1]
            name_j = "Benign" if j == 0 else attack_names[j-1]
            print(f"  {name_i:15} – {name_j:15}: {dist:.4f}")

# Summary
print("\n=== DIAGNOSTIC SUMMARY ===")
if bin_auc > 0.85 and multi_f1_macro > 0.6:
    print("✅ SSL encoder is highly discriminative for both binary and multi‑class.")
    print("   The fine‑tuning stage should work well. Focus on temporal fusion and classifier tuning.")
elif bin_auc > 0.7 and multi_f1_macro > 0.4:
    print("⚠️ SSL encoder has moderate discriminative power.")
    print("   Some classes are well‑separated, others may overlap.")
    print("   Consider: longer SSL training, different augmentations, or increasing encoder capacity.")
else:
    print("❌ SSL encoder embeddings are weak (binary AUC < 0.7 or multi‑class F1 < 0.4).")
    print("   You need to improve SSL pretraining before fine‑tuning.")
print("================================\n")

# Optionally, save results to JSON
results = {
    'binary_auc': float(bin_auc),
    'binary_f1': float(bin_f1),
    'multi_macro_f1': float(multi_f1_macro),
    'per_class': {k: {'f1': per_class_f1.get(k, 0), 'auc': per_class_auc.get(k, 0)} for k in set(per_class_f1.keys()) | set(per_class_auc.keys())}
}
with open('ssl_diagnostic_results.json', 'w') as f:
    json.dump(results, f, indent=4)
print("Diagnostic results saved to ssl_diagnostic_results.json")